In [7]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [8]:
ds = load_dataset("christinacdl/binary_hate_speech")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    3883 non-null   object
 1   label   3883 non-null   object
dtypes: object(2)
memory usage: 60.8+ KB


In [9]:
test['label'] = test['label'].apply(lambda x: 'hateful' if x == 'OFF_HATEFUL_TOXIC' else 'safe')

labels = test['label'].unique()

test

,text,label
0,i have to study... #face #pizza (i stole my ...,safe
1,days porn movie srilankanboyssex,safe
2,feeling for friends left in the place we use...,safe
3,why only target little #muslim children for mi...,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe
...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe
3880,A hoe wants attention a women wants respect.,hateful
3881,@user there is only one requirement for the jo...,hateful


In [10]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [11]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of possibly hateful content. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Tweet: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'safe' in text:
        return 'safe'
    elif 'hateful' in text:
        return 'hateful'
    else:
        return 'error'

In [12]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_binary1.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_binary1.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,i have to study... #face #pizza (i stole my ...,safe,safe,1.586730,2.0,112.0,114.0,safe
1,days porn movie srilankanboyssex,safe,hateful,0.477959,3.0,99.0,102.0,hateful
2,feeling for friends left in the place we use...,safe,safe,0.523785,2.0,118.0,120.0,safe
3,why only target little #muslim children for mi...,hateful,hateful,0.495453,3.0,118.0,121.0,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe,hateful,0.598752,3.0,138.0,141.0,hateful
...,...,...,...,...,...,...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful,hateful,0.622676,3.0,119.0,122.0,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe,hateful,0.602165,3.0,125.0,128.0,hateful
3880,A hoe wants attention a women wants respect.,hateful,hateful,0.517063,3.0,99.0,102.0,hateful
3881,@user there is only one requirement for the jo...,hateful,hateful,0.552035,3.0,122.0,125.0,hateful


In [13]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,i have to study... #face #pizza (i stole my ...,safe,safe,1.586730,2.0,112.0,114.0,safe
1,days porn movie srilankanboyssex,safe,hateful,0.477959,3.0,99.0,102.0,hateful
2,feeling for friends left in the place we use...,safe,safe,0.523785,2.0,118.0,120.0,safe
3,why only target little #muslim children for mi...,hateful,hateful,0.495453,3.0,118.0,121.0,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe,hateful,0.598752,3.0,138.0,141.0,hateful
...,...,...,...,...,...,...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful,hateful,0.622676,3.0,119.0,122.0,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe,hateful,0.602165,3.0,125.0,128.0,hateful
3880,A hoe wants attention a women wants respect.,hateful,hateful,0.517063,3.0,99.0,102.0,hateful
3881,@user there is only one requirement for the jo...,hateful,hateful,0.552035,3.0,122.0,125.0,hateful


In [14]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.722122
F1 score: 0.721884
Precision: 0.722873
Recall: 0.722122


In [15]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.6119462821767897
Average completion tokens: 2.529229976822045
Average prompt tokens: 120.2150399175895
Average total tokens: 122.74426989441153


In [16]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.07591185000000024


In [17]:
with open('results/openai_ZS_binary1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')